# Gestione del Rischio Alluvionale nelle Filippine: Segmentazione Data-Driven dei Progetti Infrastrutturali
**Corso: Principi e modelli della percezione — Prof.ssa Elena Casiraghi, Università degli Studi di Milano**
Aaliyah Elchico, Sarah Fadli


**Obiettivo:** dividere automaticamente i comuni delle Filippine in gruppi con caratteristiche simili, incrociando i dati sui progetti e sui fondi anti-alluvione con le mappe satellitari delle inondazioni reali, per verificare se le risorse sono state investite nelle zone più colpite.

Non avendo un'etichetta già pronta che classifichi i comuni come "virtuosi" o "critici", usiamo il **Clustering** (apprendimento non supervisionato) per raggruppare automaticamente i comuni simili in base a quanto si sono allagati e a quanti fondi hanno ricevuto. Questo permette di far emergere eventuali squilibri, come territori molto colpiti dall'acqua ma con pochi finanziamenti, oppure zone molto finanziate ma poco allagate.

Mettiamo insieme due fonti:
1. **Il denaro**: i progetti anti-alluvione del Ministero dei lavori pubblici filippino (DPWH), 2018–2025 (BetterGov Open Data Portal).
2. **L'acqua**: le aree allagate del 6 luglio 2025 rilevate dal radar del satellite Sentinel-1A (PhilSA).

> **Nota metodologica sull'orizzonte temporale:** i fondi DPWH rappresentano un flusso cumulato pluriennale (2018–2025), mentre il rilievo Sentinel-1A fotografa un singolo evento meteorologico (6 luglio 2025). I gruppi emersi vanno quindi letti come confronto tra *investimenti storici accumulati* e *impatto osservato in quello specifico evento*, non come misura esaustiva del rischio storico.

Per ogni comune calcoliamo quanta acqua e quanti soldi ha ricevuto, poi raggruppiamo i comuni simili con **K-Means** e controlliamo i gruppi con **Mean-Shift**.

## 0. Preparazione
Ambiente di Lavoro, Riproducibilità e Caricamento Dati

In [ ]:
#pip install numpy pandas geopandas pyarrow matplotlib seaborn scipy scikit-learn

In [2]:
import os
import io
import json
import urllib.request
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Patch
import seaborn as sns

from scipy.stats import kruskal, spearmanr, skew
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MeanShift, estimate_bandwidth
from sklearn.metrics import (
    silhouette_score,
    adjusted_rand_score,
    calinski_harabasz_score,
    davies_bouldin_score,
)
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

# 1. Riproducibilità globale, CRS metrico (Filippine) e palette cromatica semantica
SEED = 42
np.random.seed(SEED)
UTM = "EPSG:32651"   # WGS 84 / UTM Zone 51N (sistema metrico ufficiale per le Filippine)

ACQUA  = "#2B7A9B"   # Azzurro: indicatori idrogeologici / alluvionali
DENARO = "#8C5638"   # Marrone: indicatori economici / investimenti DPWH
CREMA  = "#F7F3EB"   # Sfondo / bordi neutri
COL    = ["#5C3317", "#A67B5B", "#D9CBB8", "#2B7A9B"]  # Colori ordinati per i 4 cluster

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# 2. Sorgente dati portabile (usa la cartella locale 'data' se presente, altrimenti legge da GitHub)
github_raw_url = "https://raw.githubusercontent.com/msAeliyah/Progetto_PH_FloodControl_MDP/main/data"
data_path = "data" if os.path.exists("data") else github_raw_url

def read_bytes(path):
    if path.startswith("http"):
        return urllib.request.urlopen(path).read()
    with open(path, "rb") as f:
        return f.read()

# 3. Caricamento dei dataset ed estrazione della chiave 'features' dal JSON Esri/ArcGIS
raw_json = json.loads(read_bytes(f"{data_path}/flood_control.json").decode("utf-8"))
if isinstance(raw_json, dict) and "features" in raw_json:
    df = pd.json_normalize(raw_json["features"])
    df.columns = [c.replace("attributes.", "").replace("properties.", "") for c in df.columns]
else:
    df = pd.DataFrame(raw_json)

gdf_admin2 = gpd.read_parquet(io.BytesIO(read_bytes(f"{data_path}/phl_admin2.parquet")))
gdf_admin3 = gpd.read_parquet(io.BytesIO(read_bytes(f"{data_path}/phl_admin3.parquet")))
gdf_flood  = gpd.read_parquet(io.BytesIO(read_bytes(f"{data_path}/20250706_Flood_S1A.parquet")))

# 4. Controllo dimensioni e sistemi di riferimento geografico (CRS)
print(f"Sorgente dati attiva: {data_path}")
print(f"Progetti (df):      {df.shape[0]:,} righe x {df.shape[1]} colonne")
print(f"Province (admin2):  {len(gdf_admin2):,} poligoni | CRS: {gdf_admin2.crs.to_string()}")
print(f"Comuni (admin3):    {len(gdf_admin3):,} poligoni | CRS: {gdf_admin3.crs.to_string()}")
print(f"Alluvione (flood):  {len(gdf_flood):,} poligoni | CRS: {gdf_flood.crs.to_string()}")

Matplotlib is building the font cache; this may take a moment.
Could not save font_manager cache [Errno 2] No such file or directory: 'C:\\Users\\elchi\\AppData\\Local\\matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


Sorgente dati attiva: https://raw.githubusercontent.com/msAeliyah/Progetto_PH_FloodControl_MDP/main/data
Progetti (df):      9,855 righe x 35 colonne
Province (admin2):  88 poligoni | CRS: EPSG:4326
Comuni (admin3):    1,642 poligoni | CRS: EPSG:4326
Alluvione (flood):  61,774 poligoni | CRS: EPSG:4326


## 1. Esplorazione Preliminare dei Dataset
Prima di aggregare i dati a livello comunale, ispezioniamo la struttura grezza del registro degli appalti pubblici e dei layer geografici. 

### Perché automatizzare l'analisi?
1. **Volume e frammentazione:** il file `flood_control.json` contiene **9.855 contratti** descritti da **35 variabili**, mentre il rilievo radar Sentinel-1A del 6 luglio 2025 comprende **61.774 poligoni** di aree allagate su **1.642 municipalità**, una complessità impossibile da ispezionare manualmente.
2. **Multidimensionalità:** valutare l'equità tra rischio e spesa richiede di osservare simultaneamente più variabili su scale diverse (superficie allagata, quota percentuale del territorio sommerso, numero di opere, budget totale e densità di investimento per $\text{km}^2$).
3. **Limiti della percezione visiva umana:** l'occhio umano coglie raggruppamenti per **prossimità** in 2 o 3 dimensioni, ma necessita di algoritmi di clustering per segmentare centinaia di territori in uno spazio a 5 dimensioni.

In [3]:
pd.set_option("display.max_columns", None)

print("--- ANTEPRIMA PROGETTI DPWH (prime 3 righe su 9.855) ---")
display(df.head(3))

print("\n--- ANTEPRIMA CONFINI COMUNALI (adm3) ---")
display(gdf_admin3.head(3))

print("\n--- VALORI MANCANTI SULLE VARIABILI CHIAVE DI df ---")
cols_check = [c for c in ["ContractID", "ContractCost", "Latitude", "Longitude", "Municipality", "Province", "InfraYear", "TypeofWork"] if c in df.columns]
display(df[cols_check].isna().sum().to_frame("Valori_Mancanti"))

--- ANTEPRIMA PROGETTI DPWH (prime 3 righe su 9.855) ---


,InfraYear,Region,Province,Municipality,ImplementingOffice,ProjectID,ProjectDescription,ProjectComponentID,ProjectComponentDescription,Program,TypeofWork,infra_type,Longitude,Latitude,ContractID,ABC,ContractCost,CompletionDateOriginal,CompletionYear,Contractor,ObjectId,CreationDate,Creator,EditDate,Editor,FundingYear,LegislativeDistrict,DistrictEngineeringOffice,GlobalID,ABC_String,ContractCost_String,CompletionDateActual,StartDate,geometry.x,geometry.y
0,2024,Region IV-B,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),Palawan 3rd District Engineering Office,P00824759LZ,Construction of Babuyan River Flood Control St...,P00824759LZ-CW1,Construction of Flood Mitigation Structure - C...,None,Construction of Flood Mitigation Structure,Flood Control Structures,118.89245,9.99653,24EG0058,17963475.0,17961569.07,1.720829e+12,2024,AZARRAGA CONSTRUCTION,1,1754268322388,dpwh_view,1754303523936,dpwh_view,2024,PALAWAN (THIRD LEGISLATIVE DISTRICT),Palawan 3rd District Engineering Office,79e8be69-90ab-488d-bf10-9204d41338ab,17963475,17961569.07,2024-05-23,02/15/2024,164,455
1,2024,Region IV-B,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),Palawan 3rd District Engineering Office,P00824758LZ,Construction of Flood Control Strucurue along ...,P00824758LZ-CW1,Construction of Flood Mitigation Structure - C...,None,Construction of Flood Mitigation Structure,Flood Control Structures,118.89251,9.99761,24EG0057,17370000.0,17369992.99,1.721693e+12,2024,AZARRAGA CONSTRUCTION,2,1754268322388,dpwh_view,1754303523936,dpwh_view,2024,PALAWAN (THIRD LEGISLATIVE DISTRICT),Palawan 3rd District Engineering Office,dea02c2f-d69d-4757-8d21-853e001736c5,17370000,17369992.99,2024-07-19,02/23/2024,164,455
2,2024,Region IV-B,PALAWAN,PUERTO PRINCESA CITY (CAPITAL) (PALAWAN),Palawan 3rd District Engineering Office,P00824750LZ,Construction of Babuyan River Flood Control St...,P00824750LZ-CW1,Construction of Flood Mitigation Structure - C...,None,Construction of Flood Mitigation Structure,Flood Control Structures,118.89323,9.99671,24EG0017,18335000.0,16802607.63,1.721779e+12,2024,F.F. GALANG CONSTRUCTION,3,1754268322388,dpwh_view,1754303523936,dpwh_view,2024,PALAWAN (THIRD LEGISLATIVE DISTRICT),Palawan 3rd District Engineering Office,7177ab8b-b330-40f5-bdda-44ce7823ddf8,18335000,16802607.63,2024-07-24,02/23/2024,164,455



--- ANTEPRIMA CONFINI COMUNALI (adm3) ---


,adm3_name,adm3_name1,adm3_name2,adm3_name3,adm3_pcode,adm2_name,adm2_name1,adm2_name2,adm2_name3,adm2_pcode,adm1_name,adm1_name1,adm1_name2,adm1_name3,adm1_pcode,adm0_name,adm0_name1,adm0_name2,adm0_name3,adm0_pcode,adm3_ref_n,geometry
0,Adams,None,None,None,PH0102801,Ilocos Norte,None,None,None,PH01028,Region I (Ilocos Region),None,None,None,PH01,Philippines,None,None,None,PH,Adams,"POLYGON ((120.96915 18.51012, 120.95867 18.463..."
1,Bacarra,None,None,None,PH0102802,Ilocos Norte,None,None,None,PH01028,Region I (Ilocos Region),None,None,None,PH01,Philippines,None,None,None,PH,Bacarra,"POLYGON ((120.66821 18.28705, 120.66441 18.282..."
2,Badoc,None,None,None,PH0102803,Ilocos Norte,None,None,None,PH01028,Region I (Ilocos Region),None,None,None,PH01,Philippines,None,None,None,PH,Badoc,"POLYGON ((120.47814 17.97717, 120.47851 17.976..."



--- VALORI MANCANTI SULLE VARIABILI CHIAVE DI df ---


,Valori_Mancanti
ContractID,0
ContractCost,0
Latitude,0
Longitude,0
Municipality,666
Province,0
InfraYear,0
TypeofWork,0



### 1.1 Il denaro: i progetti DPWH
Ogni riga è un contratto. Le variabili che usiamo:

| Variabile | Tipo | Significato |
|---|---|---|
| `ContractID` | categorica (identificativo) | codice del contratto |
| `Longitude`, `Latitude` | numerica continua | posizione del progetto |
| `ContractCost` | numerica continua | valore del contratto in pesos |
| `TypeofWork` | categorica nominale | tipo di opera (argine, muro di sponda, drenaggio…) |
| `InfraYear` | numerica discreta | anno |
| `Province`, `Municipality` | categorica nominale | dove si trova |

In [ ]:
P = df.copy()
print("righe totali grezze:", len(P))
P[["ContractID", "Longitude", "Latitude", "ContractCost", "TypeofWork", "InfraYear", "Province", "Municipality"]].head()

In [ ]:
print("Principali tipologie di intervento:")
display(P["TypeofWork"].value_counts().head(8).to_frame("Contratti"))

print("\nDistribuzione dei progetti e dei fondi per anno (InfraYear):")
anni = P.assign(cost_num=pd.to_numeric(P["ContractCost"], errors="coerce")).groupby("InfraYear").agg(
    n_contratti=("ContractID", "count"),
    budget_mld=("cost_num", lambda s: round(s.sum() / 1e9, 2))
)
display(anni)

Rimuoviamo i contratti duplicati e convertiamo coordinate e costi in formato numerico, **tracciando separatamente i record privi di coordinate GPS valide o con importo mancante/nullo** per non confondere un'informazione sconosciuta con uno zero verificato.

In [ ]:
n_iniziali = len(P)
P = P.drop_duplicates("ContractID").copy()
print(f"Contratti grezzi: {n_iniziali:,} | Univoci dopo deduplicazione: {len(P):,} (rimossi {n_iniziali - len(P)} duplicati)")

for c in ["Longitude", "Latitude", "ContractCost"]:
    P[c] = pd.to_numeric(P[c], errors="coerce")

# Audit sui costi: distinguiamo costi mancanti (NaN), costi pari a 0 e costi positivi
n_cost_na   = P["ContractCost"].isna().sum()
n_cost_zero = (P["ContractCost"] == 0).sum()
print(f"Contratti con ContractCost mancante (NaN): {n_cost_na} | Con ContractCost == 0 verificato: {n_cost_zero}")

# Audit sulle coordinate GPS (bounding box valido per le Filippine: Lon 116-127, Lat 4-22)
valid_gps = (
    P["Longitude"].between(116.0, 127.5) &
    P["Latitude"].between(4.0, 22.0)
)
P_no_gps = P[~valid_gps].copy()
P = P[valid_gps].copy()

print(f"Contratti con GPS valido: {len(P):,} | Esclusi per GPS mancante/fuori scala: {len(P_no_gps):,}")
print(f"Valore totale contratti geolocalizzati: {round(P['ContractCost'].sum() / 1e9, 2)} miliardi di pesos")

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 7))
gdf_admin2.plot(ax=ax, color="#EFE8DF", edgecolor="#D8CFC4", lw=0.3)
ax.scatter(P.Longitude, P.Latitude, s=1.5 + np.sqrt(P.ContractCost.fillna(0) / 1e6) * 0.9, c=DENARO, alpha=0.45, lw=0)
ax.set_xlim(116.8, 126.8); ax.set_ylim(4.5, 19.2); ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Ogni punto è un progetto DPWH (2018–2025),\nproporzionale al suo valore")

### 1.2 L'acqua: le aree allagate del 6 luglio 2025
Lo shapefile PhilSA contiene poligoni di acqua rilevata dal radar (variabile geometrica).

In [ ]:
fl = gpd.read_file(DATA + "20250706_Flood_S1A.shp")[["geometry"]]
fl["geometry"] = fl.geometry.make_valid()
print("poligoni allagati:", len(fl))

### 1.3 I comuni
I confini amministrativi (livello 3 = comuni) servono per unire le due fonti.

In [ ]:
adm3 = gpd.read_file(DATA + "phl_admin3.shp", columns=["adm3_name", "adm3_pcode", "adm2_name", "area_sqkm"])
adm3["geometry"] = adm3.geometry.simplify(0.0005)
print("comuni:", len(adm3))

## 2. Dal progetto al comune

**Filtro di copertura.** Il satellite osserva solo una parte del Paese. Un comune fuori dall'area osservata non è "non allagato": semplicemente non lo sappiamo. Il file con l'area osservata non era disponibile, quindi la **ricostruiamo come involucro convesso delle aree allagate** e teniamo solo i comuni che ci cadono dentro.

In [ ]:
hull = fl.union_all().convex_hull
adm3["in_cov"] = adm3.geometry.representative_point().within(hull)
print("comuni dentro l'area osservata:", adm3.in_cov.sum())

**Acqua per comune**: ritagliamo i poligoni allagati sui confini dei comuni e sommiamo le aree in km². *(È il passaggio più lento, qualche minuto.)*

In [ ]:
a3 = adm3[adm3.in_cov].to_crs(UTM)
inter = gpd.overlay(fl.to_crs(UTM), a3[["adm3_pcode", "geometry"]], how="intersection", keep_geom_type=True)
flood_km2 = (inter.area / 1e6).groupby(inter.adm3_pcode).sum()

**Denaro per comune**: ogni progetto va al comune in cui cadono le sue coordinate (unione spaziale). È più affidabile che confrontare i nomi, scritti in modi diversi.

In [ ]:
pg = gpd.GeoDataFrame(P, geometry=gpd.points_from_xy(P.Longitude, P.Latitude), crs="EPSG:4326")
pj = gpd.sjoin(pg, adm3[["adm3_pcode", "geometry"]], predicate="within")
agg = pj.groupby("adm3_pcode").agg(n_progetti=("ContractID", "count"), budget=("ContractCost", "sum"))
print("progetti collocati in un comune:", len(pj))

**La tabella finale**: una riga per comune, cinque misure.

In [ ]:
T = adm3[adm3.in_cov].drop(columns="geometry").set_index("adm3_pcode")
T["area_km2"] = a3.set_index("adm3_pcode").area / 1e6
T["flood_km2"] = flood_km2.reindex(T.index).fillna(0)
T = T.join(agg).fillna({"n_progetti": 0, "budget": 0})
T["quota_allagata"] = T.flood_km2 / T.area_km2
T["budget_per_km2"] = T.budget / T.area_km2
print(len(T), "comuni in", T.adm2_name.nunique(), "province")
T[["adm3_name", "adm2_name", "flood_km2", "quota_allagata", "budget", "n_progetti", "budget_per_km2"]].describe().round(2)

| Misura | Tipo | Significato |
|---|---|---|
| `flood_km2` | numerica continua | km² allagati il 6 luglio 2025 |
| `quota_allagata` | numerica continua (0–1) | parte del territorio allagata |
| `budget` | numerica continua | somma dei contratti, pesos |
| `n_progetti` | numerica discreta | numero di contratti |
| `budget_per_km2` | numerica continua | budget diviso la superficie |

### 2.1 Acqua contro denaro
Il clustering non ha una variabile *target*, ma la nostra domanda mette a confronto acqua e denaro: guardiamoli insieme.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4.5))
ax.scatter(np.log1p(T.flood_km2), np.log1p(T.budget / 1e6), s=10, c=DENARO, alpha=0.5, lw=0)
ax.set_xlabel("log(1 + km² allagati)"); ax.set_ylabel("log(1 + budget, milioni di pesos)")
r = spearmanr(T.flood_km2, T.budget)
ax.set_title(f"Spearman ρ = {r.statistic:.2f} (p = {r.pvalue:.0e})");

Il legame è **positivo ma debole**: più acqua va con un po' più di fondi, ma tanti comuni allagati ricevono poco o niente.

## 3. Normalizzazione
Due problemi:
1. **Code lunghe**: pochi comuni con valori enormi schiaccerebbero tutti gli altri. Usiamo `log(1 + x)` per le misure in km² e in pesos, e la radice quadrata per la quota allagata.
2. **Scale diverse**: km² e pesos non sono confrontabili. Con la **standardizzazione** ogni misura ha media 0 e deviazione standard 1.

In [ ]:
X = pd.DataFrame({
    "km² allagati":   np.log1p(T.flood_km2),
    "Quota allagata": np.sqrt(T.quota_allagata),
    "Budget totale":  np.log1p(T.budget / 1e6),
    "N. progetti":    np.log1p(T.n_progetti),
    "Budget per km²": np.log1p(T.budget_per_km2 / 1e6),
}, index=T.index)
prima = T[["flood_km2", "quota_allagata", "budget", "n_progetti", "budget_per_km2"]].apply(skew).round(2).values
pd.DataFrame({"asimmetria prima": prima, "asimmetria dopo": X.apply(skew).round(2).values}, index=X.columns)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].hist(T.budget / 1e6, bins=40, color="#9C8A7E"); axs[0].set_title("Prima: budget per comune (milioni di pesos)")
axs[1].hist(X["Budget totale"], bins=20, color=DENARO); axs[1].set_title("Dopo: log(1 + budget)")
Z = StandardScaler().fit_transform(X)

## 4. K-Means
Il legame con il corso: per la **Gestalt** l'occhio raggruppa gli oggetti vicini (*vicinanza*) e simili (*somiglianza*). K-Means fa lo stesso con le distanze tra comuni.

**Quanti gruppi?** Proviamo K da 2 a 8 e guardiamo il **gomito** (inerzia) e la **silhouette**.

In [ ]:
res = []
for k in range(2, 9):
    km = KMeans(k, n_init=20, random_state=42).fit(Z)
    res.append((k, km.inertia_, silhouette_score(Z, km.labels_)))
res = pd.DataFrame(res, columns=["K", "inerzia", "silhouette"])
fig, axs = plt.subplots(1, 2, figsize=(10, 3.5))
axs[0].plot(res.K, res.inerzia, "o-", color=DENARO); axs[0].set_title("Gomito"); axs[0].set_xlabel("K")
axs[1].bar(res.K, res.silhouette, color=[ACQUA if k == 4 else "#C9B8A6" for k in res.K]); axs[1].set_ylim(0.3, 0.37); axs[1].set_title("Silhouette media"); axs[1].set_xlabel("K")
res.round(3)

Dopo K = 4 il gomito si appiattisce e la silhouette è massima (0,36): scegliamo **K = 4**. Una silhouette moderata significa che i gruppi esistono ma hanno confini sfumati.

In [ ]:
K = 4
km = KMeans(K, n_init=50, random_state=42).fit(Z)
T["cluster"] = km.labels_
T.cluster.value_counts().sort_index()

## 5. Chi c'è in ogni gruppo
Valori **mediani** nella scala originale.

In [ ]:
prof = T.groupby("cluster").agg(comuni=("adm3_name", "count"),
                                 km2_allagati=("flood_km2", "median"),
                                 quota_allagata=("quota_allagata", "median"),
                                 budget_mln=("budget", lambda s: s.median() / 1e6),
                                 progetti=("n_progetti", "median"))
prof.round(3)

Leggendo la tabella diamo un nome ai gruppi. *(Con `random_state=42` i numeri dei gruppi sono quelli sotto; se cambiano, aggiornare i nomi.)*

In [ ]:
NOMI = {0: "Molti fondi, poca acqua", 1: "Acqua senza fondi", 2: "Poca acqua, pochi fondi", 3: "Molta acqua"}
T["gruppo"] = T.cluster.map(NOMI)
quote = T.groupby("gruppo").agg(acqua=("flood_km2", "sum"), denaro=("budget", "sum"))
quote = (100 * quote / quote.sum()).round(1)
quote.plot.bar(color=[ACQUA, DENARO], figsize=(7, 3.5), rot=0, title="% dell'area allagata e % dei fondi per gruppo")
quote

Il gruppo **molta acqua** ha il 77% dell'area allagata ma il 26% dei fondi. Il gruppo **molti fondi** ha il 58% dei soldi e il 3,5% dell'acqua.

In [ ]:
T[T.gruppo == "Acqua senza fondi"].sort_values("flood_km2", ascending=False)[["adm3_name", "adm2_name", "flood_km2", "n_progetti"]].head(8)

## 6. Controprova con Mean-Shift
Mean-Shift non chiede K: sposta ogni punto verso la zona più densa e trova da solo i gruppi. La larghezza di banda viene stimata dai dati (quantile 0,1).
Misuriamo l'accordo con K-Means con l'**Adjusted Rand Index** (1 = identici, 0 = a caso).

In [ ]:
bw = estimate_bandwidth(Z, quantile=0.1, random_state=42)
ms = MeanShift(bandwidth=bw, bin_seeding=True).fit(Z)
T["ms"] = ms.labels_
print("gruppi trovati da Mean-Shift:", T.ms.nunique(), "· ARI:", round(adjusted_rand_score(T.cluster, T.ms), 2))
pd.crosstab(T.gruppo, T.ms)

Per vedere i gruppi in due dimensioni usiamo la **PCA**. Le ellissi attorno ai gruppi usano il principio di **recinzione**.

In [ ]:
pca = PCA(2).fit(Z); P2 = pca.transform(Z); T["pc1"], T["pc2"] = P2[:, 0], P2[:, 1]
print("varianza spiegata:", pca.explained_variance_ratio_.round(2))
MSC = ["#C9B8A6", "#B07A52", "#6F97C2", "#34211B", "#A0BEDA"]
fig, axs = plt.subplots(1, 2, figsize=(11, 5), sharex=True, sharey=True)
for ax, lab, cols, ttl in [(axs[0], "cluster", COL, "K-Means, K = 4"), (axs[1], "ms", dict(enumerate(MSC)), "Mean-Shift")]:
    for c in sorted(T[lab].unique()):
        s = T[T[lab] == c]
        ax.scatter(s.pc1, s.pc2, s=10, color=cols[c % len(cols)], alpha=0.85, lw=0)
        if len(s) > 5:
            v, w = np.linalg.eigh(np.cov(s.pc1, s.pc2)); ang = np.degrees(np.arctan2(w[1, 1], w[0, 1]))
            ax.add_patch(Ellipse((s.pc1.mean(), s.pc2.mean()), 4 * np.sqrt(v[1]), 4 * np.sqrt(v[0]), angle=ang, fill=False, ec=cols[c % len(cols)], lw=1.2))
    ax.set_title(ttl); ax.set_xlabel("PC1 · più fondi →")
axs[0].set_ylabel("PC2 · più acqua →");

## 7. La mappa
Latitudine e longitudine **non** entrano nell'algoritmo: se i gruppi formano macchie compatte sulla mappa, hanno colto qualcosa di reale del territorio.

In [ ]:
m = adm3.merge(T[["cluster"]], left_on="adm3_pcode", right_index=True, how="left")
fig, ax = plt.subplots(figsize=(7, 8))
m[m.cluster.isna()].plot(ax=ax, color="#EFE8DF", edgecolor="#E6DDD2", lw=0.2)
for c in range(K):
    m[m.cluster == c].plot(ax=ax, color=COL[c], edgecolor=CREMA, lw=0.25, label=NOMI[c])
ax.set_xlim(118.2, 124.6); ax.set_ylim(12.0, 18.9); ax.set_aspect("equal"); ax.axis("off")
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=COL[c], label=NOMI[c]) for c in range(K)], loc="lower left", frameon=False);

## 8. Quali misure contano di più
- **Kruskal-Wallis**: per ogni misura, i gruppi sono diversi? (test non parametrico, adatto a dati non normali)
- **Random Forest + permutation importance**: un modello impara a riconoscere il gruppo; mescolando una misura alla volta vediamo quanto peggiora.

In [ ]:
H = {c: kruskal(*[X[c][T.cluster == k] for k in range(K)]) for c in X.columns}
rf = RandomForestClassifier(300, random_state=42).fit(Z, T.cluster)
pi = permutation_importance(rf, Z, T.cluster, n_repeats=20, random_state=42)
imp = pd.DataFrame({"Kruskal H": {c: h.statistic for c, h in H.items()}, "p-value": {c: h.pvalue for c, h in H.items()},
                    "permutation importance": pd.Series(pi.importances_mean, index=X.columns)}).sort_values("permutation importance")
imp["permutation importance"].plot.barh(color=[DENARO if "udget" in c or "progetti" in c else ACQUA for c in imp.index], figsize=(6, 3))
imp.sort_values("permutation importance", ascending=False).round(4)

## 9. Conclusioni
- **Il denaro segue l'acqua solo in parte.** I comuni più allagati hanno il 77% dell'acqua ma il 26% dei fondi; i comuni con più fondi si allagano poco.
- Esistono **comuni molto allagati senza nessun progetto** (es. Allacapan e Tuao in Cagayan).
- K-Means e Mean-Shift concordano sui gruppi principali (ARI ≈ 0,6).
- A separare i gruppi sono prima di tutto il **budget**, poi la **quota allagata**.

**Limiti**
- Un solo evento alluvionale: un comune può essere a rischio anche se il 6 luglio non si è allagato.
- L'area osservata dal satellite è ricostruita da noi.
- "Zero progetti" può significare progetti senza coordinate.
- Il budget dice quanto si spende, non se le opere funzionano.

In [ ]:
T.to_csv("comuni_cluster.csv")